In [ ]:
from google.colab import drive
import importlib
import os
import sys

drive.mount('/content/drive')
REPO_DIR = '/content/drive/MyDrive/pyrosense-si'
os.chdir(REPO_DIR)
if os.path.join(REPO_DIR, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
importlib.invalidate_caches()
print('Working directory:', os.getcwd())

In [ ]:
!pip install -q -e .
import torch

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from pyrosense.config import DATA_PROCESSED_DIR
from pyrosense.data import fetch_psi_s3_manifest

manifest = fetch_psi_s3_manifest(DATA_PROCESSED_DIR / 'psi_s3_manifest.json', max_keys_per_prefix=40)
print('Indexed S3 objects:', len(manifest.get('sampled_objects', [])))
print('NTRS combustion papers:', len(manifest.get('ntrs_combustion_papers', [])))
print('NASA flame images:', len(manifest.get('nasa_image_library_flames', [])))

In [ ]:
from pyrosense.config import DATA_PROCESSED_DIR, DATA_RAW_DIR, MODELS_CHECKPOINT_DIR
from pyrosense.models import train_pinn

pinn_res = train_pinn(
    csv_path=DATA_RAW_DIR / 'psi_burn_runs.csv',
    model_out=MODELS_CHECKPOINT_DIR / 'pinn_flammability.pt',
    grid_out=DATA_PROCESSED_DIR / 'pinn_phase_grid.json',
    epochs=2500,
    lr=3e-3,
)
print('Trained on:', pinn_res['gpu_name'])
print('Validation metrics:', pinn_res['validation_metrics'])

In [ ]:
from pyrosense.config import DATA_PROCESSED_DIR, MODELS_CHECKPOINT_DIR, WEB_FRAMES_DIR
from pyrosense.models.pde_field_pinn import train_2d_reacting_flow_pinn

pde_res = train_2d_reacting_flow_pinn(
    checkpoint_out=MODELS_CHECKPOINT_DIR / 'pinn_2d_reacting_flow.pt',
    json_out=DATA_PROCESSED_DIR / 'pinn_2d_fields.json',
    image_out=WEB_FRAMES_DIR / 'pinn_2d_fields.png',
    epochs=2400,
    batch_colloc=8192,
    lr=2e-3,
)
for s in pde_res['regime_field_summaries']:
    print(s)

In [ ]:
from pyrosense.config import DATA_PROCESSED_DIR, MODELS_CHECKPOINT_DIR, WEB_FRAMES_DIR
from pyrosense.vision import run_flame_cv_pipeline
from pyrosense.vision.unet_trainer import train_flame_unet

cv_res = run_flame_cv_pipeline(
    frames_dir=WEB_FRAMES_DIR,
    metrics_out=DATA_PROCESSED_DIR / 'flame_morphology_metrics.json',
)
unet_res = train_flame_unet(
    checkpoint_out=MODELS_CHECKPOINT_DIR / 'flame_unet.pt',
    metrics_out=DATA_PROCESSED_DIR / 'unet_vision_metrics.json',
    strip_img_out=WEB_FRAMES_DIR / 'unet_segmentation_strip.png',
    epochs=80,
    batch_size=16,
    lr=2e-3,
)
print('CV regimes:', len(cv_res['regimes']), '| FlameUNet final val mIoU:', unet_res['final_val_miou'])